# 4장 체인의 구조를 고도화하기, Runnable
### 필요 패키지 설치
- 아래 코드 셀을 실행하여 필요한 패키지를 설치합니다.

In [ ]:
!pip install -U langchain langchain-openai

In [ ]:
!pip show langchain
!pip show langchain-openai

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "오픈AI API Key"

## 4.1 RunnableParallel
> langchain_core.runnables 모듈의 RunnableParallel 클래스를 사용하여 두 가지 이상의 Pipeline을 병렬적으로 실행 가능

#### 개별 체인 정의
- 병렬로 실행할 체인을 정의합니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI


prompt1 = PromptTemplate.from_template("{topic}를 주제로 농담 하나 해줘")
prompt2 = PromptTemplate.from_template("{topic}를 주제로 시 하나 작성해 줘")

model = ChatOpenAI(model="gpt-5-nano")

parser = StrOutputParser()

joke_chain = prompt1 | model | parser
poem_chain = prompt2 | model | parser

#### RunnableParallel 객체 생성 및 체인 할당
- `RunnableParallel` 클래스를 불러와 객체를 생성합니다

In [ ]:
from langchain_core.runnables import RunnableParallel

map_chain = RunnableParallel(joke=joke_chain, poem=poem_chain)

#### 체인 실행  


In [ ]:
map_chain.invoke({"topic": "친구"})

#### 실습 RunnableParallel

요구사항 1: 영화 질문 답변 및 Prompt 생성
1. 사용자는 영화와 관련된 질문을 입력합니다.
2. Chain 1 을 통해 사용자 질문에 대한 답변(answer)을 받습니다.
3. RunnableParallel을 이용하여 두 작업을 동시에 수행합니다:
    * Chain 2-1 : 추천 프롬프트(prompt_recommend):
        * 사용자의 질문과 AI의 답변을 바탕으로 사용자가 새롭게 질문할 만한 추천 프롬프트 3개를 생성하시오.
    * Chain 2-2 : 유사 질문(sim_question):
        * 사용자의 질문과 유사하지만 사용된 단어가 다른 새로운 질문 3개를 생성하시오.

요구사항 2: 리스트 형식으로 반환
1. 2-1, 2-2 Chain은 모두 CommaSeparatedListOutputParser를 사용해 리스트 형태로 반환합니다.
2. 최종 출력은 {"prompt_recommend": [...], "sim_question": [...]} 형태여야 합니다.


In [ ]:
question = "인셉션 감독이 누구인가요?"

In [ ]:
template1 = """
당신은 영화를 추천해주는 AI 챗봇입니다. User의 질문에 대해 답변하시오.
question : {question}
"""

prompt = PromptTemplate(
    template=template1
)

model = ChatOpenAI(model_name="gpt-5-nano")

chain1 = prompt | model | StrOutputParser()

answer = chain1.invoke({"question" : question})

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()
# 출력 형식 지침
format_instructions = output_parser.get_format_instructions()


recommend_template = """
당신은 영화를 추천해 주는 AI 챗봇입니다. User의 질문과 AI의 답변 다음으로 User가 질문할 만한 추천 프롬프트 3개를 말해주세요.
User_Message : {question}
AI_Message : {answer}

FORMAT :
{format}
"""

recommend_prompt = PromptTemplate(
    template=recommend_template,
    partial_variables={
        "format": format_instructions
    },
)

augmented_template = """
당신은 사용자의 질문을 새롭게 생성하는 인공지능 비서입니다. User의 질문과 유사하지만 사용한 단어는 다른 질문 3개를 말해주세요.
User_Message : {question}

FORMAT :
{format}
"""

augmented_prompt = PromptTemplate(
    template=augmented_template,
    partial_variables={
        "format": format_instructions
    },
)

model = ChatOpenAI()


recommend_response = recommend_prompt | model | output_parser
augmented_question = augmented_prompt | model | output_parser

In [ ]:
from langchain_core.runnables import RunnableParallel

map_chain = RunnableParallel(
    recommend_response=recommend_response,
    augmented_question=augmented_question)

result = map_chain.invoke({"question" : question, "answer" : answer})

In [ ]:
result

---

## 4.2 RunnableLambda-기초

> RunnableLambda를 사용하여 사용자 정의 함수를 Chain에 맵핑할 수 있다.

#### 일반 함수와 lambda 함수

In [ ]:
# 일반 함수
def add(x, y):
    return x + y

# lambda 함수 예제
numbers = [1, 2, 3, 4]
double_numbers = list(map(lambda x: x * 2, numbers))
print(double_numbers) # [2, 4, 6, 8]

### RunnableLambda 1
#### 함수 정의
- RunnableLambda에서 파이썬 기본 내장 함수 len을 바로 사용할 수도 있지만, 예시를 위함이니 간단하게 정의합니다.

In [ ]:
def length_function(word):
    return len(word)

#### 기초 체인 정의
- 기본 프롬프트, 모델, 출력 파서를 정의합니다.

In [ ]:
prompt = PromptTemplate.from_template(
    "{a} + {b}는 무엇인가요?"
)
model = ChatOpenAI()
output_parser = StrOutputParser()

#### RunnableLambda를 통한 함수 연결 및 실행


In [ ]:
from langchain_core.runnables import RunnableLambda

# Chain 구성
chain = (
    {
        "a": RunnableLambda(lambda x: length_function(x["word1"])),
        "b": RunnableLambda(lambda x: length_function(x["word2"])),
    }
    | prompt
    | model
    | output_parser
)

chain.invoke({"word1": "안녕하세요", "word2": "반가워요"})

### RunnableLambda를 활용한 라우팅
#### classifier_chain 체인 정의
- 사용자의 질문에 대한 주제를 파악해서 한 단어로 답변하는 체인

In [ ]:
template = """
주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요.
이외의 답변은 허용하지 않습니다.
<question>
{question}
</question>

<answer example>
수학
</answer example>
"""

classifier_prompt = PromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-5-nano")
output_parser = StrOutputParser()

classifier_chain = classifier_prompt | model | output_parser

result = classifier_chain.invoke({"question": "2+2 는 무엇인가요?"})

result

#### 개별 체인 생성
- 각 주제에 특화된 math_chain, science_chain, general_chain을 정의합니다.

In [ ]:
# 수학 체인
math_prompt = PromptTemplate.from_template(
        """
          당신은 수학 전문가입니다.
          항상 다음과 같이 답변을 시작합니다. "피타고라스께서 말씀하시기를…"


          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )

# 과학 체인
science_prompt = PromptTemplate.from_template(
        """
          당신은 과학 전문가입니다.\
          항상 다음과 같이 답변을 시작합니다. "뉴턴께서 말씀하시기를…"

          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )

# 일반 체인
general_prompt = PromptTemplate.from_template(
        """
          당신은 일반 상식 전문가입니다.\
          항상 다음과 같이 답변을 시작합니다. "부모님께서 말씀하시기를 …"

          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )


math_chain = math_prompt | model | output_parser
science_chain = science_prompt | model | output_parser
general_chain = general_prompt | model | output_parser

#### Route 함수 정의
- classifier_chain의 결과(주제)를 입력받아 조건에 따라 분기 처리하는 route 함수를 정의합니다.
- 단, 이때 이 route 함수가 언제 호출될 것이며, 이때 전달되는 값이 어떤 형식인지를 생각하여 정의해야 합니다.

In [ ]:
def route(info):
    if "수학" in info["topic"]:
        return math_chain
    elif "과학" in info["topic"]:
        return science_chain
    else:
        return general_chain

#### full chain 연결
- 앞에서 정의한 각종 함수와 체인들을 모두 결합하여 full_chain을 정의합니다.

In [ ]:
full_chain = (
     {"topic": classifier_chain}
    | RunnableLambda(route)
    | StrOutputParser()
)

full_chain.invoke({"question": "2+2는 뭔가요?"})

#### KeyError 해결 후 최종 코드

In [ ]:
# 전체 체인 정의
full_chain = (
     {
         "topic": classifier_chain,
         "question": lambda x: x["question"]
      }
    | RunnableLambda(route)
    | StrOutputParser()
)

full_chain.invoke({"question": "2+2는 뭔가요?"})

#### 실습: RunnableLambda-router

사용자가 질문을 던지면, 챗봇은 이를 **유튜브, 동물병원, 보험, 기타**로 분류한 후 적절한 체인으로 라우팅하여 응답합니다.  RunnableLambda, PromptTemplate, 및 체인 연결을 활용하여 동작하는 AI 시스템을 구현하시오.

**요구사항**
1. 입력 데이터:
    * 사용자는 질문(question)을 입력합니다.
2. 라우터:
    * 주어진 질문을 유튜브, 동물병원, 보험, 기타 중 하나로 분류해야 합니다.
    * router_chain에서 질문을 분석하고 결과를 topic 키의 값으로 반환합니다.
3. 라우팅 로직:
    * **RunnableLambda**를 사용하여 topic 값에 따라 적절한 체인을 선택하세요:
        * 유튜브 → youtube_chain
        * 동물병원 → hospital_chain
        * 보험 → insurance_chain
        * 기타 → general_chain
4. 최종 결과 출력:
    * 선택된 체인의 결과를 문자열로 반환하며, 최종 응답을 출력합니다.


In [ ]:
template = """주어진 사용자 질문을 `유튜브`, `동물병원`, `보험` 또는 `기타` 중 하나로 분류하세요. 한 단어 이상으로 응답하지 마세요.
<question>
{question}
</question>

Classification:
"""

classifier_prompt = PromptTemplate.from_template(template)

classifier_chain = classifier_prompt | ChatOpenAI(model="gpt-5-nano") | StrOutputParser()

In [ ]:
# router_chain의 invoke한 값에 따라 chain 호출
def route(info):
    if "유튜브" in info["topic"]:
        return youtube_chain
    elif "동물병원" in info["topic"]:
        return hospital_chain
    elif "보험" in info["topic"]:
        return insurance_chain
    else:
        return general_chain

In [ ]:
# 유튜브 체인
youtube_chain = (
    PromptTemplate.from_template(
        """
          당신은 유튜브 영상을 추천하는 인공지능 챗봇입니다..\
          언제나 다음과 같이 답변을 시작하시오 "당신의 알고리즘 기반 추천 영상은 다음과 같습니다..". \
          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )
    | ChatOpenAI(model="gpt-5-nano")
)

# 동물병원 체인
hospital_chain = (
    PromptTemplate.from_template(
        """
          당신은 동물병원을 추천하는 인공지능 챗봇입니다..\
          언제나 다음과 같이 답변을 시작하시오 "인근 동물병원을 찾아볼게요..". \
          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )
    | ChatOpenAI(model="gpt-5-nano")
)

# 보험 체인
insurance_chain = (
    PromptTemplate.from_template(
        """
          당신은 보험 기반 상담 챗봇입니다..\
          언제나 다음과 같이 답변을 시작하시오 "가입한 보험 기반 답변해드릴 게요..". \
          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )
    | ChatOpenAI(model="gpt-5-nano")
)

# 일반 체인
general_chain = (
    PromptTemplate.from_template(
        """
          당신은 강아지 안구 진단 전문가 입니다.\
          언제나 다음과 같이 답변을 시작하시오 "제 개인적인 견해로는..". \
          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )
    | ChatOpenAI(model="gpt-5-nano")
)

In [ ]:
# 전체 체인 정의
full_chain = (
    {
        "topic": classifier_chain,
        "question": lambda x: x["question"]
    }
    | RunnableLambda(route)
    | StrOutputParser()
)

In [ ]:
full_chain.invoke({"question": "괜찮은 유튜브 영상 추천해 주세요"})

In [ ]:
full_chain.invoke({"question": "서울역 근처 동물병원 추천해 주세요."})

In [ ]:
full_chain.invoke({"question": "현대해상 펫보험은 예금자 보호 상품인가요?"})

In [ ]:
full_chain.invoke({"question": "백내장에 좋은 약은 뭐가 있나요?"})

---

## 4.3 RunnablePassthrough
- 원활한 실습 진행을 위한 classifier_chain 재정의

- 아래 코드를 먼저 실행하고 다음 과정으로 넘어갑니다.


In [ ]:

template = """
주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요.
이외의 답변은 허용하지 않습니다.
<question>
{question}
</question>

<answer example>
수학
</answer example>
"""

classifier_prompt = PromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-5-nano")
output_parser = StrOutputParser()

classifier_chain = classifier_prompt | model | output_parser

result = classifier_chain.invoke({"question": "2+2 는 무엇인가요?"})

result

#### 데이터 흐름 테스트
- 마찬가지로, 아래 코드를 실행하여 전체 데이터의 흐름을 파악합니다.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

full_chain = (
    {
      "topic": classifier_chain,
      "question": lambda x: x["question"]
    }
    | RunnableParallel(
        passed = RunnablePassthrough(),
        modified = RunnableLambda(route)
    )
)

full_chain.invoke({"question": "2+2는 뭔가요?"})

#### RunnablePassthrough.assign()

In [ ]:
full_chain = (
    RunnablePassthrough.assign(topic=classifier_chain)
    | RunnableLambda(route)
    | StrOutputParser()
)

full_chain.invoke({"question": "2+2는 뭔가요?"})